Upload file

In [6]:
import os
from google.colab import files

#  Use Kaggle.json
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Please upload your kaggle.json")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

#  Dataset

!kaggle datasets download -d isareskhamwirai/data-fromsuper-ai-engineer-ss6-heart-disease
!unzip -q *.zip -d data/

Dataset URL: https://www.kaggle.com/datasets/isareskhamwirai/data-fromsuper-ai-engineer-ss6-heart-disease
License(s): CC-BY-SA-4.0
data-fromsuper-ai-engineer-ss6-heart-disease.zip: Skipping, found more recently modified local copy (use --force to force download)


Unzip

In [7]:
!unzip -o *.zip -d data

Archive:  data-fromsuper-ai-engineer-ss6-heart-disease.zip
  inflating: data/sample_submission.csv  
  inflating: data/test.csv           
  inflating: data/train.csv          


Check

In [8]:
import os
print(os.listdir("data"))

['test.csv', 'sample_submission.csv', 'train.csv']


Step 1. Data Cleansing & Preprocessing

ในขั้นตอนแรก เราจะทำการทำความสะอาดข้อมูล (Data Cleansing) เนื่องจากชุดข้อมูลที่ได้รับมีความไม่สมบูรณ์หลายจุด (Missing Values) และมีข้อมูลบางส่วนที่เป็นข้อความ (Categorical Data) ซึ่งต้องแปลงเป็นตัวเลขเพื่อให้โมเดลสามารถประมวลผลได้

* การจัดการข้อมูลที่หายไป (Handling

Missing Values)
เราเลือกใช้กลยุทธ์ที่เหมาะสมกับธรรมชาติของข้อมูลแต่ละประเภท:

Target Column: หากแถวใดไม่มีค่าใน History of HeartDisease or Attack จะถูกลบทิ้งทันที เนื่องจากเราไม่สามารถนำข้อมูลที่ไม่มีฉลาก (Label) มาเทรนโมเดลได้

Numerical Features (เช่น BMI): ใช้ค่า Median (ค่ากลาง) ในการแทนที่ค่าว่าง เพื่อลดผลกระทบจากค่าที่สูงหรือต่ำผิดปกติ (Outliers)

Categorical Features: ใช้ค่า Mode (ฐานนิยม) หรือค่าที่ปรากฏบ่อยที่สุดในการแทนที่ค่าว่าง

* การแปลงข้อมูลกลุ่มเป็นตัวเลข (Feature Encoding)
เพื่อให้โมเดลสามารถคำนวณทางคณิตศาสตร์ได้ เราแบ่งการจัดการออกเป็น 2 รูปแบบ:

Binary Encoding: แปลงค่า Yes/No และ Male/Female ให้เป็น 1 และ 0

Ordinal Mapping: สำหรับข้อมูลที่มีลำดับความสำคัญ เช่น General Health (Excellent > Poor) เราจะใช้การ Mapping ด้วยตัวเลข (5-0) เพื่อรักษาความสัมพันธ์ของระดับข้อมูลไว้

Step 2: Model Training with Cross-Validation

* เราเลือกใช้เทคนิค Stratified 5-Fold
Cross-Validation เพื่อป้องกันการเกิด Overfitting และเพื่อให้มั่นใจว่าโมเดลสามารถทำงานได้ดีกับข้อมูลที่หลากหลาย โดยใช้โมเดล LightGBM ซึ่งมีประสิทธิภาพสูงกับข้อมูลประเภทตาราง (Tabular Data)

* Optimization Strategy
Class Weight 'balanced': เนื่องจากข้อมูลโรคหัวใจมักจะมีจำนวนคนที่เป็นโรคน้อยกว่าคนปกติ (Imbalanced Data) เราจึงให้ความสำคัญกับกลุ่มที่เป็นโรคมากขึ้น

* Early Stopping: ป้องกันโมเดลเรียนรู้มากเกินไปจนจำคำตอบ (Overfit) โดยจะหยุดเทรนหากผลลัพธ์ใน Validation Set ไม่ดีขึ้นติดต่อกัน 50 รอบ

In [17]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import fbeta_score

# ==========================================
# 1. LOAD DATA & CLEANING
# ==========================================
print("📦 Loading and Cleaning Data...")

# โหลดไฟล์ (ปรับ Path ให้ตรงกับที่ unzip ไว้)
# ถ้าคุณ unzip ลงโฟลเดอร์ data ให้ใช้ 'data/train.csv'
try:
    train = pd.read_csv('data/train.csv')
    test = pd.read_csv('data/test.csv')
except:
    train = pd.read_csv('train.csv')
    test = pd.read_csv('test.csv')

target_col = 'History of HeartDisease or Attack'

# ลบแถวที่ไม่มีเฉลย (Target)
train = train.dropna(subset=[target_col]).reset_index(drop=True)

# Mapping ค่าต่าง ๆ
binary_map = {'Yes': 1, 'No': 0, 'Male': 1, 'Female': 0}
health_map = {'Excellent': 5, 'Very Good': 4, 'Good': 3, 'Fair': 2, 'Poor': 1, 'Very Poor': 0}
education_map = {'College graduate': 4, 'Some college or technical school': 3, 'High school graduate': 2, 'Some high school': 1, 'Elementary': 0}
income_map = {'$75,000 or more': 7, '$50,000 to less than $75,000': 6, '$35,000 to less than $50,000': 5, '$25,000 to less than $35,000': 4, '$20,000 to less than $25,000': 3, '$15,000 to less than $20,000': 2, '$10,000 to less than $15,000': 1, 'Less than $10,000': 0}

binary_cols = ['History of HeartDisease or Attack', 'High Blood Pressure', 'Told High Cholesterol',
               'Cholesterol Checked', 'Smoked 100+ Cigarettes', 'Diagnosed Stroke',
               'Diagnosed Diabetes', 'Leisure Physical Activity', 'Heavy Alcohol Consumption',
               'Health Care Coverage', 'Doctor Visit Cost Barrier', 'Difficulty Walking',
               'Sex', 'Vegetable or Fruit Intake (1+ per Day)']

for df in [train, test]:
    for col in binary_cols:
        if col in df.columns:
            df[col] = df[col].map(binary_map)
            m = df[col].mode()
            df[col] = df[col].fillna(m[0] if not m.empty else 0)

    if 'General Health' in df.columns:
        df['General Health'] = df['General Health'].map(health_map).fillna(3)
    if 'Education Level' in df.columns:
        df['Education Level'] = df['Education Level'].map(education_map).fillna(2)
    if 'Income Level' in df.columns:
        df['Income Level'] = df['Income Level'].map(income_map).fillna(3)

    for num_col in ['Body Mass Index', 'Age']:
        if num_col in df.columns:
            df[num_col] = pd.to_numeric(df[num_col], errors='coerce')
            df[num_col] = df[num_col].fillna(df[num_col].median())

print(f"✅ Cleaned! Train shape: {train.shape}, Test shape: {test.shape}")

# ==========================================
# 2. PREPARE X, y & TRAINING
# ==========================================
X = train.drop(columns=['ID', target_col])
y = train[target_col].astype(int) # มั่นใจว่าเป็น Integer
test_df = test.drop(columns=['ID'])

print(f"🛠️ X shape: {X.shape}, y shape: {y.shape}")

if X.shape[0] == 0:
    print("❌ ERROR: Data is still empty. Please check if train.csv is loaded correctly.")
else:
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(test_df))

    lgbm_params = {
        'n_estimators': 1000, 'learning_rate': 0.05, 'max_depth': 7, 'num_leaves': 31,
        'class_weight': 'balanced', 'metric': 'binary_logloss', 'random_state': 42, 'verbosity': -1
    }

    print("🔥 Starting Training...")
    for i, (train_idx, val_idx) in enumerate(kf.split(X, y)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = lgb.LGBMClassifier(**lgbm_params)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50)])

        oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
        test_preds += model.predict_proba(test_df)[:, 1] / kf.get_n_splits()
        print(f"✅ Fold {i+1} Done")

📦 Loading and Cleaning Data...
✅ Cleaned! Train shape: (221390, 20), Test shape: (74361, 19)
🛠️ X shape: (221390, 18), y shape: (221390,)
🔥 Starting Training...
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's binary_logloss: 0.435217
✅ Fold 1 Done
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's binary_logloss: 0.432308
✅ Fold 2 Done
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's binary_logloss: 0.435732
✅ Fold 3 Done
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's binary_logloss: 0.435911
✅ Fold 4 Done
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's binary_logloss: 0.43682
✅ Fold 5 Done


Step 3: Threshold Optimization (F2-Score)
ในโจทย์ทางการแพทย์ การทำนายผิดว่า "ไม่เป็นโรค" ทั้งที่จริงๆ "เป็นโรค" (False Negative) มีความเสี่ยงสูงมาก เราจึงเลือกใช้ F2-Score เป็นเกณฑ์ในการตัดสินใจ ซึ่งจะให้น้ำหนักกับ Recall มากกว่า Precision

เราจะทำการค้นหา Best Threshold ที่ดีที่สุดจากผลลัพธ์ของ Out-of-Fold (OOF) เพื่อนำไปใช้ในการทำนายผลลัพธ์สุดท้าย (Final Submission)

In [18]:
# ==========================================
# 3. FIND BEST THRESHOLD (F2-Score)
# ==========================================

best_threshold = 0.5
best_f2 = 0

# วนลูปหา Threshold ที่ดีที่สุดในช่วง 0.1 - 0.6
print("\n🎯 Optimizing Threshold for F2-Score...")
for t in np.arange(0.1, 0.6, 0.01):
    # คำนวณ F2-Score (ให้น้ำหนัก Recall มากกว่า Precision)
    score = fbeta_score(y, (oof_preds > t).astype(int), beta=2)
    if score > best_f2:
        best_f2 = score
        best_threshold = t

print(f"🔥 Best Threshold: {best_threshold:.2f}")
print(f"📈 Best OOF F2-Score: {best_f2:.4f}")

# ==========================================
# 4. FINAL SUBMISSION
# ==========================================

# ใช้ Best Threshold ที่หาได้มาทำนาย Test Set
final_pred = (test_preds > best_threshold).astype(int)

# แปลงกลับเป็นข้อความ 'Yes' / 'No' ตาม Format โจทย์
final_label = pd.Series(final_pred).map({0: 'No', 1: 'Yes'})

submission = pd.DataFrame({
    "ID": test['ID'],
    target: final_label
})

submission.to_csv("submission.csv", index=False)
print("\n✅ Successfully saved 'submission.csv'")


🎯 Optimizing Threshold for F2-Score...
🔥 Best Threshold: 0.54
📈 Best OOF F2-Score: 0.5347

✅ Successfully saved 'submission.csv'
